# 13 — SHAP Explainability

Explains the selected machine-learning model using SHAP values.

Install SHAP first if needed:

```bash
pip install shap
```

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap

data = pd.read_csv(PROCESSED_DIR / "station_samples" / "station_raster_samples.csv")
bundle = joblib.load(MODEL_DIR / "random_forest.joblib")
pipeline = bundle["pipeline"]
features = bundle["features"]

X = data[features].replace([np.inf, -np.inf], np.nan)
X_transformed = pipeline.named_steps["imputer"].transform(X)
model = pipeline.named_steps["model"]

sample_size = min(1000, len(X))
sample_index = np.random.default_rng(42).choice(
    len(X), size=sample_size, replace=False
)
X_sample = X_transformed[sample_index]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, feature_names=features)